In [1]:
from pathlib import Path

import boto3
import pandas as pd
import numpy as np

from botocore.exceptions import BotoCoreError, ClientError

from secretkeys import ACCESS_KEY, KEY_ID

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Обучение модели

Для начала загрузим данные:

In [2]:
S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/train.csv"
LOCAL_PATH = Path("data/train.csv")

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=KEY_ID,
    aws_secret_access_key=ACCESS_KEY,
    region_name="ru-central1",
)

def read_s3_csv(bucket, key):

    response = s3.get_object(Key=key, Bucket=bucket)
    return pd.read_csv(response['Body'])

def read_data(bucket, key, local_path=None):
   
   try:
      return read_s3_csv(bucket, key)
   except (BotoCoreError, ClientError, OSError) as e:
      if local_path:
         print(f'Reading data failed with {type(e).__name__}')
         print(f'Attempting local data at {local_path}')
         try:
            df = pd.read_csv(local_path)
            return df
         except FileNotFoundError as e:
            raise e 
      else:
         raise e

In [3]:
df = read_data(S3_BUCKET, S3_KEY, LOCAL_PATH)

В данных есть 35 пропущенных значений. Их мы заполним при подготовки данных к обучению. Также видим, что есть несколько категориальных колонок, много численных. Еще мы удалим snapshot date, тк она одинаковая у всех, а также id, тк, наоборот, они у всех разные.

In [4]:
df = df.drop(columns=['snapshot_date', 'listing_id'])
df.isna().sum()

latitude                   0
longitude                  0
accommodates               0
bathrooms                  6
bedrooms                   7
beds                      21
minimum_nights             1
neighbourhood_cleansed     0
property_type              0
room_type                  0
price                      0
dtype: int64

In [186]:
df.nunique()

latitude                  5396
longitude                 5515
accommodates                16
bathrooms                   19
bedrooms                    12
beds                        16
minimum_nights              12
neighbourhood_cleansed      33
property_type               51
room_type                    4
price                     2480
dtype: int64

Начнем с двух фич, которые мы можем добавить, чтобы потенциально улучшить предсказания: количества ванных на количество спален, и расстояния до центра Лондона, т.к. neighbourhood именно из Лондона.

In [5]:
df['bathroom_to_bedroom_ratio'] = df['bathrooms'] / df['bedrooms'].replace(0, np.nan)
df['distance_to_centre'] = ((df['latitude'] - 51.5074) ** 2 + (df['longitude'] - 0.1278))

Мы будем решать задачу регрессии и предсказывать цену. Разделяем данные на трейн и тест. 

In [6]:
X, y = df.drop(columns=['price']), df['price']

X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=42)

Разделим признаки на категориальные и на численные. Заполним пропуски и масштабируем численные признаки. Также логарифмируем таргет, потому что у нас есть достаточно сильно отличающиеся от нормы выбросы в таргете.

In [7]:
def log_target(y):
    return np.log1p(y)

def rev_target(y):
    return np.expm1(y)

y_train = log_target(y_train)

In [8]:
nums = ['latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', \
        'beds', 'minimum_nights', 'bathroom_to_bedroom_ratio', 'distance_to_centre']
cats = ['neighbourhood_cleansed', 'property_type', 'room_type']

for col in nums:
    med =  X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_val[col] = X_val[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

    aveg, scale = X_train[col].mean(), X_train[col].std(ddof=0)
    X_train[col] = (X_train[col] - aveg) / scale
    X_val[col] = (X_val[col] - aveg) / scale
    X_test[col] = (X_test[col] - aveg) / scale

Заполним пропуски и закодируем категориальные признаки с помощью get dummies.

In [9]:
for col in cats:
    med = X_train[col].mode()
    X_train[col] = X_train[col].fillna(med)
    X_val[col] = X_val[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

X_train = pd.get_dummies(X_train, columns=cats, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=cats, drop_first=True, dtype=int).reindex(columns=X_train.columns, fill_value=0)
X_val =  pd.get_dummies(X_val, columns=cats, drop_first=True, dtype=int).reindex(columns=X_train.columns, fill_value=0)

Обучим модель линейной регрессии в качестве бейзлайна. Сначала без параметров.

In [10]:
model = LinearRegression()
model.fit(X_train, y_train)
preds = model.predict(X_val)

Оценим r2 и MAE ошибку. Почему я использую MAE: у нас есть очень высокие, аномальные значения таргета, которые могут искусственно раздувать метрику MSE из-за квадратов. R2 здесь вторичен, потому что он чувствителен к выбросам тоже, но может помочь определиться, если метрика MAE будет схожей.

In [11]:
def scores(y_true, y_pred):
    mse = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'MAE = {mse:.3f}')
    print(f'R2 = {r2:.3f}')

scores(y_val, rev_target(preds))

MAE = 86.322
R2 = 0.182


Это достаточно высокое значение ошибки. R2 тоже низкий.

Теперь подберем параметры для нескольких разных моделей с помощью GridSearch:

In [12]:
def gridsearch(model, grid):
    grid = GridSearchCV(estimator=model, param_grid=grid, cv=5, scoring='neg_mean_absolute_error')
    return grid

models = [Ridge(random_state=42), Lasso(random_state=42), SVR(), RandomForestRegressor(random_state=42), GradientBoostingRegressor(loss='absolute_error', random_state=42)]
labels = ['Ridge', 'Lasso', 'SVR', 'Random Forest', 'Gradient Boosting']

In [13]:
grid_ridge = {
    "alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["auto", "svd", "lsqr"],
}

grid_lasso = {
    "alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

grid_svr = {
    'C': [0.1, 10, 100],
    'epsilon': [0.01, 0.1, 0.5]
}

grid_rf = {
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 2, 4]
}

grid_gbr = {
    "n_estimators": [200, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5]
}

params = [grid_ridge, grid_lasso, grid_svr, grid_rf, grid_gbr]

In [14]:
results = []

for label, model, param in zip(labels, models, params):
    print(label)
    grid = gridsearch(model, param)
    grid.fit(X_train, y_train)
    y_pred = rev_target(grid.best_estimator_.predict(X_val))
    scores(y_val, y_pred)
    results.append(grid.best_estimator_)
    print('')

Ridge
MAE = 86.055
R2 = 0.183

Lasso
MAE = 86.392
R2 = 0.188

SVR
MAE = 85.104
R2 = 0.145

Random Forest
MAE = 82.191
R2 = 0.235

Gradient Boosting
MAE = 81.175
R2 = 0.209



Random Forest и Gradient Boosting немножко улучшили наши предсказания (но не сильно). А на каких айтемах делаются ошибки?

In [40]:
mistakes = pd.DataFrame()
mistakes['target'] = y_val
mistakes.reset_index(inplace=True, drop=True)
mistakes['predicted'] = pd.Series(rev_target(results[4].predict(X_val)))
mistakes['error'] = np.abs(mistakes['target'] - mistakes['predicted'])
mistakes['diff'] = mistakes['error'] / mistakes['target']
mistakes['is_outlier'] = mistakes['target'] > np.quantile(mistakes['target'], 0.99)
mistakes['is_lower_error'] = mistakes['diff'] < 0.2

In [41]:
mistakes.sort_values(by='error', ascending=False).head(10)

,target,predicted,error,diff,is_outlier,is_lower_error
702,5706.00,96.835920,5609.164080,0.983029,True,False
311,5024.50,196.775905,4827.724095,0.960837,True,False
976,4440.50,385.848514,4054.651486,0.913107,True,False
630,2400.00,876.686440,1523.313560,0.634714,True,False
663,2073.33,970.901872,1102.428128,0.531719,True,False
160,1556.43,746.886249,809.543751,0.520129,True,False
253,1026.03,247.312421,778.717579,0.758962,True,False
579,828.00,87.028181,740.971819,0.894894,False,False
906,789.00,87.623788,701.376212,0.888943,False,False
56,900.00,234.114018,665.885982,0.739873,False,False


In [44]:
mistakes[mistakes.is_lower_error == True]['is_outlier'].value_counts()

is_outlier
False    472
Name: count, dtype: int64

In [47]:
mistakes[mistakes.is_outlier == False]['diff'].mean()

np.float64(0.3281039732734433)

Модель, как и ожидалось, сильно шибается на выбросах: значениях таргета выше 99-ого перцентиля. Причем серьезно ошибается: в первом случае аж на 98 процентов, что из-за большого числа самого таргета сильно портит нам ошибку. Везде, где модель ошибается не сильно, таргет не выброс.

В части с пайпланом мы попробуем добавить NLP. А пока что закончим и протестируем модель на тесте:

In [46]:
y_pred = rev_target(results[4].predict(X_test))
scores(y_test, y_pred)

MAE = 69.609
R2 = 0.494


### Выводы:

Мы добавили два новых признака, обработали старые, обучили бейзлайн модель и несколько других с grid search. Лучшей моделью оказалась Gradient Boosting, которая выдала 81 MAE и 0.2 R2 на валидационном сете. Самыми "ошибочными" предсказаниями были выбросы. На остальных значениях ошибка была около 33% процентов в среднем, и на половине датасета модель ошибалась менее, чем на 20%. Таким образом, большую часть ошибки составляли именно аномальные значения таргета. На тестовой выборке модель достигла 70 MAE и 0.5 R2.